# DeepSeek SFT on Google Colab (40 GB GPU)

This notebook installs the repository's SFT environment and runs **only** the `deepseek` profile (`deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`). The training implementation remains in `trainer/sft`; the notebook only configures Colab and invokes the existing CLI.

Before running it:

1. Select a GPU runtime in **Runtime > Change runtime type**. The defaults below target one nominal 40 GB GPU.
2. Add `HF_TOKEN` and `WANDB_API_KEY` in Colab's **Secrets** panel (key icon). Do not paste tokens into notebook cells.
3. The defaults run 300 optimizer steps, print training metrics every 50 steps, and run the complete validation split once at step 300. Per-batch training and evaluation progress bars are disabled so they do not overwhelm the Colab page.

Checkpoints and the final LoRA are uploaded through the repository's existing W&B workflow, which is important because a Colab VM is temporary.

In [ ]:
# Repository and run controls
REPO_URL = "https://github.com/Haeryz/Putusan-crawler.git"  # @param {type:"string"}
REPO_BRANCH = "master"  # @param {type:"string"}
REPO_DIR = "/content/Sinergi"  # @param {type:"string"}

TRAIN_STEPS = 300  # @param {type:"integer"}
EVAL_EVERY_STEPS = 300  # @param {type:"integer"}
SAVE_EVERY_STEPS = 50  # @param {type:"integer"}
LOG_EVERY_STEPS = 50  # @param {type:"integer"}
WANDB_RUN_NAME = "colab-deepseek-sft"  # @param {type:"string"}

# Conservative 40 GB profile. 6 x 4 preserves the repository's effective batch of 24.
MICRO_BATCH_SIZE = 6  # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 4  # @param {type:"integer"}
MAX_SEQUENCE_LENGTH = 8192  # @param {type:"integer"}

from pathlib import Path
import os
import subprocess

repo_dir = Path(REPO_DIR).expanduser().resolve()
if not (repo_dir / "pyproject.toml").is_file():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(f"{repo_dir} exists but is not a Sinergi checkout")
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)],
        check=True,
    )

sft_main = repo_dir / "trainer" / "sft" / "main.py"
if not sft_main.is_file():
    raise RuntimeError(f"Missing expected SFT entry point: {sft_main}")

os.environ["SINERGI_REPO_DIR"] = str(repo_dir)
os.chdir(repo_dir)
print(f"Using repository: {repo_dir}")

In [ ]:
# Load credentials without storing them in the notebook.
from google.colab import userdata

def require_colab_secret(name: str) -> str:
    try:
        value = userdata.get(name)
    except Exception as error:
        raise RuntimeError(f"Add {name} in the Colab Secrets panel and enable notebook access") from error
    if not value:
        raise RuntimeError(f"Colab secret {name} is empty")
    return value

os.environ["HF_TOKEN"] = require_colab_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = require_colab_secret("WANDB_API_KEY")
os.environ["HF_HOME"] = "/content/huggingface-cache"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("HF_TOKEN and WANDB_API_KEY loaded from Colab Secrets.")

In [ ]:
# Verify that Colab assigned the requested GPU class before installing packages.
query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip().splitlines()
if not query:
    raise RuntimeError("No NVIDIA GPU detected; select a GPU runtime")

gpu_name, memory_mib = [part.strip() for part in query[0].rsplit(",", 1)]
GPU_VRAM_GIB = float(memory_mib) / 1024
print(f"GPU: {gpu_name} ({GPU_VRAM_GIB:.2f} GiB)")
if GPU_VRAM_GIB < 38:
    raise RuntimeError(
        "This notebook's defaults target a nominal 40 GB GPU (at least 38 GiB reported). "
        "Choose a larger runtime or deliberately lower the batch/context settings."
    )

## Install the DeepSeek SFT environment

The repository's root dependencies do not include the ML stack. This cell creates a Python 3.12 virtual environment and installs the versions used by `trainer/sft/setup_runpod.sh`, while omitting the Qwen/Gemma-specific model download and the incompatible 80 GB deep preflight. Re-running the cell reuses the existing environment.

In [ ]:
%%bash
set -euo pipefail
cd "${SINERGI_REPO_DIR}"
python -m pip install --quiet uv
VENV_PY="${SINERGI_REPO_DIR}/.venv/bin/python"
if [[ ! -x "${VENV_PY}" ]]; then
  uv venv --python 3.12 "${SINERGI_REPO_DIR}/.venv"
fi
uv pip install --python "${VENV_PY}" "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes "xformers==0.0.32.post2"
uv pip install --python "${VENV_PY}" "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
uv pip install --python "${VENV_PY}" --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" "unsloth>=2026.4.2" unsloth_zoo
uv pip install --python "${VENV_PY}" "transformers>=5.5.0,<5.6" "trl>=0.28.0,<0.29" "huggingface_hub>=1.5.0" "datasets==4.3.0"
# TorchAO 0.17+ requires PyTorch 2.11 APIs; 0.16 supports this repo's Torch 2.8 pin.
uv pip install --python "${VENV_PY}" --no-deps "torchao==0.16.0"
uv pip install --python "${VENV_PY}" wandb numpy tqdm peft accelerate sentencepiece protobuf packaging
uv pip install --python "${VENV_PY}" --editable .
"${VENV_PY}" -c "import torch, torchao; from transformers import AutoConfig, AutoProcessor, AutoTokenizer; print(f'torch={torch.__version__}, torchao={torchao.__version__}, CUDA={torch.version.cuda}, GPUs={torch.cuda.device_count()}')"

In [ ]:
# Fast DeepSeek-only service/schema preflight. This intentionally does not load Gemma.
venv_python = repo_dir / ".venv" / "bin" / "python"
preflight_env = os.environ.copy()
preflight_env["PYTHONUNBUFFERED"] = "1"
preflight = subprocess.run(
    [str(venv_python), "-m", "trainer.sft.preflight", "--model", "deepseek"],
    cwd=repo_dir,
    env=preflight_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(preflight.stdout, end="")
if preflight.returncode != 0:
    raise RuntimeError(
        "DeepSeek preflight failed. The specific dependency, HF_TOKEN, dataset, "
        "or WANDB_API_KEY error is printed immediately above this message."
    )

## Train and evaluate

This calls the existing `trainer.sft` module. `--allow-non-a100` bypasses the repository's 80 GB hardware guard; the cell independently checks the requested configuration against the repository's conservative memory estimator using the GPU's reported VRAM. Evaluation uses a per-device batch size of 1, runs once at step 300, and has no per-batch progress bar. Training metrics and checkpoint messages appear every 50 steps.

In [ ]:
from dataclasses import replace
import shlex
import sys

for name, value in {
    "TRAIN_STEPS": TRAIN_STEPS,
    "EVAL_EVERY_STEPS": EVAL_EVERY_STEPS,
    "SAVE_EVERY_STEPS": SAVE_EVERY_STEPS,
    "LOG_EVERY_STEPS": LOG_EVERY_STEPS,
    "MICRO_BATCH_SIZE": MICRO_BATCH_SIZE,
    "GRADIENT_ACCUMULATION_STEPS": GRADIENT_ACCUMULATION_STEPS,
    "MAX_SEQUENCE_LENGTH": MAX_SEQUENCE_LENGTH,
}.items():
    if not isinstance(value, int) or isinstance(value, bool) or value < 1:
        raise ValueError(f"{name} must be a positive integer; got {value!r}")

sys.path.insert(0, str(repo_dir))
from trainer.sft.config import MODEL_PROFILES, run_config_for_model
from trainer.sft.memory import estimate_training_memory

profile = run_config_for_model(MODEL_PROFILES["deepseek"])
training = replace(
    profile.training,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)
estimate = estimate_training_memory(
    profile.model, training, MAX_SEQUENCE_LENGTH, GPU_VRAM_GIB
)
print(
    f"Conservative estimated peak: {estimate.total_gib:.2f} GiB; "
    f"safe budget: {estimate.usable_vram_gib:.2f} GiB"
)
if estimate.total_gib > estimate.usable_vram_gib:
    raise RuntimeError(
        "Requested batch/context does not fit the repository's safe memory budget; "
        "lower MICRO_BATCH_SIZE or MAX_SEQUENCE_LENGTH."
    )

cli_args = [
    "--modelname", "deepseek",
    "--gpu-count", "1",
    "--allow-non-a100",
    "--max-steps", str(TRAIN_STEPS),
    "--eval-steps", str(EVAL_EVERY_STEPS),
    "--save-steps", str(SAVE_EVERY_STEPS),
    "--max-seq-length", str(MAX_SEQUENCE_LENGTH),
    "--per-device-batch-size", str(MICRO_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
]
if WANDB_RUN_NAME.strip():
    cli_args.extend(["--wandb-run-name", WANDB_RUN_NAME.strip()])

# Keep the repository CLI unchanged while overriding its existing logging_steps field.
runner_code = r'''
from dataclasses import replace
import os
import sys
from transformers import PrinterCallback, ProgressCallback
from trainer.sft.cli import build_parser, config_from_args
import trainer.sft.main as training_workflow

training_workflow.load_training_env()
args = build_parser().parse_args(sys.argv[1:])
config = config_from_args(args)
config = replace(
    config,
    training=replace(
        config.training,
        logging_steps=int(os.environ["SINERGI_LOG_EVERY_STEPS"]),
    ),
)
original_build_trainer = training_workflow.build_trainer
def build_quiet_trainer(*build_args, **build_kwargs):
    trainer = original_build_trainer(*build_args, **build_kwargs)
    trainer.remove_callback(ProgressCallback)
    trainer.remove_callback(PrinterCallback)
    trainer.add_callback(PrinterCallback)
    return trainer
training_workflow.build_trainer = build_quiet_trainer
training_workflow.run_training(config)
'''
command = [str(venv_python), "-u", "-c", runner_code, *cli_args]
display_command = [str(venv_python), "-m", "trainer.sft", *cli_args]
print("Running:", shlex.join(display_command), flush=True)
print(
    f"Quiet output: train metrics/checkpoints every {LOG_EVERY_STEPS} steps; "
    f"one validation at step {EVAL_EVERY_STEPS}.",
    flush=True,
)
training_env = os.environ.copy()
training_env["PYTHONUNBUFFERED"] = "1"
training_env["SINERGI_LOG_EVERY_STEPS"] = str(LOG_EVERY_STEPS)
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["TQDM_DISABLE"] = "1"
process = subprocess.Popen(
    command,
    cwd=repo_dir,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    assert process.stdout is not None
    for line in process.stdout:
        # Drop any residual tqdm redraws from model/data downloads or evaluation.
        if "\r" in line or ("%|" in line and ("it/s" in line or "s/it" in line)):
            continue
        print(line, end="", flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    print("Stopping the DeepSeek training subprocess...", flush=True)
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)